## Purpose
- Creates input X data for the model - matrix with major allele variation info for each isolate
- Genotype phenotype map dataframe that has ground truth phenotypes for each isolate

In [1]:
import numpy as np
from scipy import stats
import pandas as pd
import time
import os

In [2]:
from evcouplings.align import Alignment # install this using 'pip install evcouplings'
import glob

### File paths

In [3]:
# master_phenotype_file = "../../MTB-CNN/input_data/phenotype/master_table_resistance.csv"
# original_fastas_dir = "../../MTB-CNN/input_data/fasta_files/combined_fastas"
# reduced_fastas_dir = "../input_data/fasta_files/reduced"
# reduced_fastas_kept_indices_dir = "../input_data/fasta_files/reduced/kept_indices"
# prepared_data_dir = "../input_data/reduced_prepared_data"

master_phenotype_file = "/project/pi_annagreen_umass_edu/saishradha/project_data_curation/phenotype_data/master_resistance_table.csv"
original_fastas_dir = "/project/pi_annagreen_umass_edu/saishradha/project_data_curation/genomic_data/aligned"
reduced_fastas_dir = "../input_data/fasta_files/reduced"
reduced_fastas_kept_indices_dir = "../input_data/fasta_files/reduced/kept_indices"
prepared_data_dir = "../input_data/reduced_prepared_data"

### Prepare directories

In [4]:
# Create the directory if it doesn't exist
os.makedirs(prepared_data_dir, exist_ok=True)
os.makedirs(reduced_fastas_dir, exist_ok=True)
os.makedirs(f"{reduced_fastas_dir}/fastas", exist_ok=True)
os.makedirs(reduced_fastas_kept_indices_dir, exist_ok=True)

print(f"Directory '{prepared_data_dir}' created or already exists.")
print(f"Directory '{reduced_fastas_dir}' created or already exists.")
print(f"Directory '{reduced_fastas_dir}/fastas' created or already exists.")
print(f"Directory '{reduced_fastas_kept_indices_dir}' created or already exists.")

Directory '../input_data/reduced_prepared_data' created or already exists.
Directory '../input_data/fasta_files/reduced' created or already exists.
Directory '../input_data/fasta_files/reduced/fastas' created or already exists.
Directory '../input_data/fasta_files/reduced/kept_indices' created or already exists.


### Read phenotype data

In [7]:
# Read in list of isolates
phenotypes =  pd.read_csv(master_phenotype_file, low_memory=False)
isolate_order = list(phenotypes.New_ID)

### Reorder the alignments to match the phenotype isolate ordering

It has been observed that for the new fastas, the MAF filtering is reducing the size of the alignments drastically. 

This is because the MAF filtering is done on the first alignment and then the other alignments are reordered to match the first alignment. This is causing the other alignments to have a lot of missing data. 

In [8]:
# Iterate through each fasta file, select only the positions with a high enough MAF
# Keep only the isolates with phenotype data, reorder the alignment to match isolate_order
# save a reduced fasta file

for f in glob.glob(f"{original_fastas_dir}/*.fasta"):
    print("Reading alignment from file:", f)
    # Reads the alignment file
    aln = Alignment.from_file(open(f))
    
    # Impose a minor allele frequency threshold of 0.001 in order to keep computation tractable
    # NOTE: the order in which you do the MAF thresholding and then subsetting the isolates matters!
    # Here it's fine because the only isolates in my fasta files are the ones that have phenotypes
    indices_to_keep = np.where(aln.frequencies.max(axis=1) < .999)[0]
    subset_alignment = aln.select(indices_to_keep)
    print(f"indices to keep: {len(indices_to_keep)}")
    print("original alignment shape", aln.matrix.shape)
    print("reduced alignment shape", subset_alignment.matrix.shape)
    # Cleanup giant variables
    del aln

    # Get a list of the ids we are keeping, in order
    
    # First, correct the ids in the alignment so that they match the table of phenotypes
    subset_alignment.ids = [x.split("/")[-1].split(".")[0] for x in subset_alignment.ids]
    subset_alignment.id_to_index = {x:idx for idx,x in enumerate(subset_alignment.ids)}
    
    # Get the indices that would correctly reorder the alignment to match isolate_order
    reorder_index = [
        subset_alignment.id_to_index[x] for x in isolate_order if x in subset_alignment.id_to_index
    ]
    
    # Reorder based on reorder_index
    subset_alignment.ids = list(np.array(subset_alignment.ids)[reorder_index])
    subset_alignment.matrix = subset_alignment.matrix[reorder_index, :]
    print("shape with isolates ordered", subset_alignment.matrix.shape)
    print("\n")
    
    # Get the name of the fasta file for saving
    name = f.split("/")[-1].split(".")[0]
    
    # save the reduced file
    subset_alignment.write(open(f"{reduced_fastas_dir}/fastas/{name}_reduced.fa", "w"))
    np.save(f"{reduced_fastas_kept_indices_dir}/{name}_indices.npy", indices_to_keep)
    
# Save the isolate indices in order, just in case
print("Saving isolate indices to file:", f"{reduced_fastas_dir}/isolate_indexes.csv")
pd.DataFrame(subset_alignment.ids).to_csv(f"{reduced_fastas_dir}/isolate_indexes.csv")

# change line 425 in alignment library: update the str

Reading alignment from file: /project/pi_annagreen_umass_edu/saishradha/project_data_curation/genomic_data/aligned/pncA.fasta
indices to keep: 765
original alignment shape (17943, 1067)
reduced alignment shape (17943, 765)
shape with isolates ordered (17942, 765)


Reading alignment from file: /project/pi_annagreen_umass_edu/saishradha/project_data_curation/genomic_data/aligned/tlyA.fasta
indices to keep: 9
original alignment shape (17943, 1035)
reduced alignment shape (17943, 9)
shape with isolates ordered (17942, 9)


Reading alignment from file: /project/pi_annagreen_umass_edu/saishradha/project_data_curation/genomic_data/aligned/ethA.fasta
indices to keep: 1088
original alignment shape (17943, 1737)
reduced alignment shape (17943, 1088)
shape with isolates ordered (17942, 1088)


Reading alignment from file: /project/pi_annagreen_umass_edu/saishradha/project_data_curation/genomic_data/aligned/ethR.fasta
indices to keep: 751
original alignment shape (17943, 751)
reduced alignment sh

### Encode the fasta files into one-hot format for learning

In [9]:
### If using one-hot, here's the functions

# Mapping to use for one-hot encoding
BASE_TO_COLUMN = {'A': 0, 'C': 1, 'T': 2, 'G': 3, '-': 4}

# Get one hot vector
def make_one_hot(matrix, BASE_TO_COLUMN=BASE_TO_COLUMN):
    n_bases = len(BASE_TO_COLUMN.keys()) 
    one_hot = np.zeros((matrix.shape[0], matrix.shape[1]* n_bases),dtype=np.int8)
    print("starting from shape", one_hot.shape)
    
    for idx in range(matrix.shape[0]):
        for jdx in range(matrix.shape[1]):
            base = matrix[idx, jdx]
            if base in BASE_TO_COLUMN:
                one_hot[idx, jdx * n_bases + BASE_TO_COLUMN[base]] = 1
            # If base is not in BASE_TO_COLUMN, it remains 0 (no action needed)

    return one_hot


In [10]:
# get the list of file names in the reduced fastas directory
# reduced_fasta_files = glob.glob(f"{reduced_fastas_dir}/fastas/*.fa")
reduced_fasta_files = [
    file for file in glob.glob(f"{reduced_fastas_dir}/fastas/*.fa")
    if os.path.basename(file) != "Rv0678_reduced.fa"
]
# ignoring Rv0678 because it's a single base pair long

# files = ['fastas_reduced/rpoBC_reduced.fa',
#  'fastas_reduced/rpsL_reduced.fa',
#  'fastas_reduced/gyrBA_reduced.fa',
#  'fastas_reduced/clpC_reduced.fa',
#  'fastas_reduced/embCAB_reduced.fa',
#  'fastas_reduced/rrs-rrl_reduced.fa',
#  'fastas_reduced/pncA_reduced.fa',
#  'fastas_reduced/panD_reduced.fa',
#  'fastas_reduced/tlyA_reduced.fa',
#  'fastas_reduced/FabG1-inhA_reduced.fa',
#  'fastas_reduced/KatG_reduced.fa',
#  'fastas_reduced/gid_reduced.fa',
#  'fastas_reduced/eis_reduced.fa',
#  'fastas_reduced/oxyR-ahpC_reduced.fa',
#  'fastas_reduced/ethAR_reduced.fa',
#  'fastas_reduced/aftB-ubiA_reduced.fa',
#  'fastas_reduced/acpM-kasA_reduced.fa',
#  'fastas_reduced/rpsA_reduced.fa']


In [14]:
# #### Make combined matrix of all the positions we'll be learning on

# Compute the total number of sites in our model by summing the length of all the alignment
total_sites = 0
for f in reduced_fasta_files:
    print(f)
    aln = Alignment.from_file(open(f))
    total_sites += aln.L
print("total sites", total_sites)
total_seqs = aln.N

../input_data/fasta_files/reduced/fastas/pncA_reduced.fa
../input_data/fasta_files/reduced/fastas/tlyA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethR_reduced.fa
../input_data/fasta_files/reduced/fastas/katG_reduced.fa
../input_data/fasta_files/reduced/fastas/eis_reduced.fa
../input_data/fasta_files/reduced/fastas/gid_reduced.fa
../input_data/fasta_files/reduced/fastas/rpsL_reduced.fa
../input_data/fasta_files/reduced/fastas/inhA_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoB_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoC_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrB_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrA_reduced.fa
../input_data/fasta_files/reduced/fastas/embC_reduced.fa
../input_data/fasta_files/reduced/fastas/embA_reduced.fa
../input_data/fasta_files/reduced/fastas/embB_reduced.fa
../input_data/fasta_files/reduced/fastas/rrs_reduced.fa
../input_data/fasta_files/reduced/

### Creating matrix that contains info about variation from the major allele at each site

In [21]:
# Matrix to store the data for learning
X = np.zeros((total_seqs, total_sites), dtype=np.int8)

current_index = 0

for f in reduced_fasta_files:
    print(f)
    aln = Alignment.from_file(open(f), alphabet='-ACGT')
    
    # Tells you which character is the most frequent in each site
    who_is_max = np.argmax(aln.frequencies, axis=1)
    
    # Major allele is encoded as 0, minor allele(s) as 1
    major_minor = aln.matrix_mapped != who_is_max
    
    # Add the encoding to the X matrix
    X[:, current_index:(current_index + major_minor.shape[1])] = major_minor
    
    # Keep track of how many sites in X we have filled in
    current_index = current_index + major_minor.shape[1]

print(f"Saving input X to {prepared_data_dir}/combined_X.npy")
np.save(f"{prepared_data_dir}/combined_X.npy", X)
print("Total seqs vs total sites:", X.shape)

../input_data/fasta_files/reduced/fastas/pncA_reduced.fa
../input_data/fasta_files/reduced/fastas/tlyA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethR_reduced.fa
../input_data/fasta_files/reduced/fastas/katG_reduced.fa
../input_data/fasta_files/reduced/fastas/eis_reduced.fa
../input_data/fasta_files/reduced/fastas/gid_reduced.fa
../input_data/fasta_files/reduced/fastas/rpsL_reduced.fa
../input_data/fasta_files/reduced/fastas/inhA_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoB_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoC_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrB_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrA_reduced.fa
../input_data/fasta_files/reduced/fastas/embC_reduced.fa
../input_data/fasta_files/reduced/fastas/embA_reduced.fa
../input_data/fasta_files/reduced/fastas/embB_reduced.fa
../input_data/fasta_files/reduced/fastas/rrs_reduced.fa
../input_data/fasta_files/reduced/

### Ref-alt encoding for capturing variations (use either is or the previous one)

In [25]:
X = np.zeros((total_seqs, total_sites), dtype=np.int8)

current_index = 0

for f in reduced_fasta_files:
    print(f)
    aln = Alignment.from_file(open(f), alphabet='-ACGT')

    # Tells you which character is the most frequent in each site
    who_is_max = np.argmax(aln.frequencies, axis=1)

    # The first sequence is the reference
    ref_seq = aln.matrix_mapped[0]  # shape: (num_sites,)

    # Encode: 1 if different from reference, 0 if same
    ref_alt = (aln.matrix_mapped != ref_seq)

    # Add encoding to the big X matrix
    X[:, current_index:(current_index + ref_alt.shape[1])] = ref_alt.astype(np.int8)

    current_index += ref_alt.shape[1]

print(f"Saving input X to {prepared_data_dir}/combined_X.npy")
np.save(f"{prepared_data_dir}/combined_X.npy", X)
print("Total seqs vs total sites:", X.shape)


../input_data/fasta_files/reduced/fastas/pncA_reduced.fa
../input_data/fasta_files/reduced/fastas/tlyA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethA_reduced.fa
../input_data/fasta_files/reduced/fastas/ethR_reduced.fa
../input_data/fasta_files/reduced/fastas/katG_reduced.fa
../input_data/fasta_files/reduced/fastas/eis_reduced.fa
../input_data/fasta_files/reduced/fastas/gid_reduced.fa
../input_data/fasta_files/reduced/fastas/rpsL_reduced.fa
../input_data/fasta_files/reduced/fastas/inhA_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoB_reduced.fa
../input_data/fasta_files/reduced/fastas/rpoC_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrB_reduced.fa
../input_data/fasta_files/reduced/fastas/gyrA_reduced.fa
../input_data/fasta_files/reduced/fastas/embC_reduced.fa
../input_data/fasta_files/reduced/fastas/embA_reduced.fa
../input_data/fasta_files/reduced/fastas/embB_reduced.fa
../input_data/fasta_files/reduced/fastas/rrs_reduced.fa
../input_data/fasta_files/reduced/

### Create a dataFrame as a mapping table of each gene and its corresponding informative site indices within the reduced fasta files.

In [26]:
# Make a table of all of the sites in the model for later mapping
# Note that the sites listed here are indexed within each fasta file - NOT the MTB genome
total_sites = []
genes = []

for f in reduced_fasta_files:
    print(f)
    
    gene_name = f.split(f"{reduced_fastas_dir}/fastas")[-1].split("_")[0]

    numpy_file = reduced_fastas_kept_indices_dir + gene_name + "_indices.npy"
    print(numpy_file)
    
    # numpy_file = f.split("reduced.fa")[0] + "indices.npy"

    sites = np.load(numpy_file)
    
    sites = sorted(list(sites))
    
    total_sites += list(sites)
    genes += [gene_name] * len(list(sites))

print("Total genes:", len(set(genes)))
print("Total sites:", len(total_sites))
    
df = pd.DataFrame({
    "locus": genes,
    "sites": total_sites,
})

df.to_csv(f"{prepared_data_dir}/site_indices.csv")
print("Saved site indices to", f"{prepared_data_dir}/site_indices.csv")

../input_data/fasta_files/reduced/fastas/pncA_reduced.fa
../input_data/fasta_files/reduced/kept_indices/pncA_indices.npy
../input_data/fasta_files/reduced/fastas/tlyA_reduced.fa
../input_data/fasta_files/reduced/kept_indices/tlyA_indices.npy
../input_data/fasta_files/reduced/fastas/ethA_reduced.fa
../input_data/fasta_files/reduced/kept_indices/ethA_indices.npy
../input_data/fasta_files/reduced/fastas/ethR_reduced.fa
../input_data/fasta_files/reduced/kept_indices/ethR_indices.npy
../input_data/fasta_files/reduced/fastas/katG_reduced.fa
../input_data/fasta_files/reduced/kept_indices/katG_indices.npy
../input_data/fasta_files/reduced/fastas/eis_reduced.fa
../input_data/fasta_files/reduced/kept_indices/eis_indices.npy
../input_data/fasta_files/reduced/fastas/gid_reduced.fa
../input_data/fasta_files/reduced/kept_indices/gid_indices.npy
../input_data/fasta_files/reduced/fastas/rpsL_reduced.fa
../input_data/fasta_files/reduced/kept_indices/rpsL_indices.npy
../input_data/fasta_files/reduced/fa

### Combine the X data with the phenotype resistance profile isolate per drug - final X and y data

In [27]:
## Create combined genotype phenotype table
phenotypes =  pd.read_csv(master_phenotype_file, index_col=0, low_memory=False)

# Convert phenotypes to numeric
y_all_rs = phenotypes.fillna('-1')
y_all_rs = y_all_rs.astype(str)
resistance_categories = {'R': 0, 'S': 1, '-1.0': -1, '-1': -1}

y_all = y_all_rs.copy()
for key, val in resistance_categories.items():
    y_all[y_all_rs == key] = val

# Fill missing data with nans
y_all[y_all==-1] = np.nan
phenotypes=y_all

# read in X matrix and isolate indexes 
X_all = np.load(f"{prepared_data_dir}/combined_X.npy")

# Create a dataframe of X's with one row per isolate
indices = pd.read_csv(f"{reduced_fastas_dir}/isolate_indexes.csv", index_col=0)
X_df = pd.concat([indices, pd.DataFrame(X_all)], axis=1)

# Add column names corresponding to position
column_names = pd.read_csv(f"{prepared_data_dir}/site_indices.csv", index_col=0)
colnames = ["isolate"] + [f"{x}_{y}" for x,y in zip(column_names.locus, column_names.sites)]
X_df.columns = colnames

# Merge with phenotypes and save
X_df = X_df.merge(phenotypes, right_on="New_ID", left_on="isolate", how="left")
print("Combined geno pheno data shape:", X_df.shape)
print("Number of isolates :", X_df.shape[0])
print("Total number of columns:", X_df.shape[1])

print("Saving to", f"{prepared_data_dir}/combined_geno_pheno_df.csv")
X_df.to_csv(f"{prepared_data_dir}/combined_geno_pheno_df.csv")

Combined geno pheno data shape: (17942, 3943)
Number of isolates : 17942
Total number of columns: 3943
Saving to ../input_data/reduced_prepared_data/combined_geno_pheno_df.csv
